<a href="https://colab.research.google.com/github/Jeancjudy/PLP6235C/blob/main/genome_assembly_simple_virus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Genome Assembly 1
Guided Assembly vs. De Novo Assembly

This notebook demonstrates various approaches to assembling genomes from raw reads. First, we’ll perform a guided assembly using recovered Illumina reads from a metagenomic study, specifically assembling begomovirus genomes. Second, we’ll conduct a de novo assembly without a reference.

##Install dependencies and tools##

**Install miniconda**

In [ ]:
# @title
!wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
!chmod +x Miniconda3-latest-Linux-x86_64.sh
!bash ./Miniconda3-latest-Linux-x86_64.sh -b -f -p /usr/local
import sys
sys.path.append('/usr/local/lib/python3.7/site-packages/')
!conda config --add channels defaults
!conda config --add channels bioconda
!conda config --add channels conda-forge

--2026-03-04 20:31:54--  https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
Resolving repo.anaconda.com (repo.anaconda.com)... 104.16.191.158, 104.16.32.241, 2606:4700::6810:bf9e, ...
Connecting to repo.anaconda.com (repo.anaconda.com)|104.16.191.158|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 162067098 (155M) [application/octet-stream]
Saving to: ‘Miniconda3-latest-Linux-x86_64.sh’

Miniconda3-latest-L 100%[===================>] 154.56M   281MB/s    in 0.5s    

2026-03-04 20:31:54 (281 MB/s) - ‘Miniconda3-latest-Linux-x86_64.sh’ saved [162067098/162067098]

PREFIX=/usr/local
Unpacking bootstrapper...
Unpacking payload...

Installing base environment...

Preparing transaction: ...working... done
Executing transaction: ...working... done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best results, please

**Install fastqc, bwa, samtools, datasets, pilon and quast**

In [ ]:
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r


accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r


In [ ]:
# @title
!conda install bioconda::bwa -y
!conda install bioconda::samtools -y
!conda install -c conda-forge ncbi-datasets-cli -y
!conda install bioconda::pilon -y
!conda install bioconda::quast -y

Jupyter detected...
2 channel Terms of Service accepted
Retrieving notices: - \ done
Channels:
 - conda-forge
 - bioconda
 - defaults
Platform: linux-64
Solving environment: | / - done

## Package Plan ##

  environment location: /usr/local

  added / updated specs:
    - bioconda::bwa


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    bwa-0.7.19                 |       h577a1d6_1         639 KB  bioconda
    ca-certificates-2026.2.25  |       hbd8a1cb_0         144 KB  conda-forge
    certifi-2026.2.25          |     pyhd8ed1ab_0         148 KB  conda-forge
    libxcrypt-4.4.36           |       hd590300_1          98 KB  conda-forge
    openssl-3.6.1              |       h35e630c_1         3.0 MB  conda-forge
    perl-5.32.1                | 7_hd590300_perl5        12.7 MB  conda-forge
    ------------------------------------------------------------
                           

In [1]:
!apt-get update -qq
!apt-get install -y fastqc

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  at-spi2-core default-jre default-jre-headless fonts-dejavu-core
  fonts-dejavu-extra gsettings-desktop-schemas libapache-pom-java
  libargs4j-java libatk-bridge2.0-0 libatk-wrapper-java
  libatk-wrapper-java-jni libatk1.0-0 libatk1.0-data libatspi2.0-0
  libcommons-compress-java libcommons-io-java libcommons-jexl2-java
  libcommons-lang3-java libcommons-logging-java libcommons-math3-java
  libcommons-parent-java libfindbin-libs-perl libhtsjdk-java libjbzip2-java
  libjson-simple-java libngs-java libngs-sdk-dev libngs-sdk2 libsis-base-java
  libsis-base-jni libsis-jhdf5-java libsis-jhdf5-jni libsnappy-java
  libsnappy-jni libxcomposite1 libxtst6 libxxf

# Assisted Assembly Using a Reference
Perform a guided assembly using recovered Illumina reads from a metagenomic study, focusing specifically on assembling begomovirus genomes.

The FASTQ files contain the reads classified as begomovirus. We will focus exclusively on these reads for our analysis."

In [2]:
!wget https://raw.githubusercontent.com/PlantHealth-Analytics/Genome_assembly/main/field_1.fastq
!wget https://raw.githubusercontent.com/PlantHealth-Analytics/Genome_assembly/main/field_2.fastq

--2026-03-04 20:36:44--  https://raw.githubusercontent.com/PlantHealth-Analytics/Genome_assembly/main/field_1.fastq
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 18838873 (18M) [text/plain]
Saving to: ‘field_1.fastq’

field_1.fastq       100%[===================>]  17.97M  --.-KB/s    in 0.1s    

2026-03-04 20:36:44 (130 MB/s) - ‘field_1.fastq’ saved [18838873/18838873]

--2026-03-04 20:36:44--  https://raw.githubusercontent.com/PlantHealth-Analytics/Genome_assembly/main/field_2.fastq
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Len

**Run Fastqc for quality control**

In [3]:
!fastqc *.fastq

Started analysis of field_1.fastq
Approx 5% complete for field_1.fastq
Approx 10% complete for field_1.fastq
Approx 15% complete for field_1.fastq
Approx 20% complete for field_1.fastq
Approx 25% complete for field_1.fastq
Approx 30% complete for field_1.fastq
Approx 35% complete for field_1.fastq
Approx 40% complete for field_1.fastq
Approx 45% complete for field_1.fastq
Approx 50% complete for field_1.fastq
Approx 55% complete for field_1.fastq
Approx 60% complete for field_1.fastq
Approx 65% complete for field_1.fastq
Approx 70% complete for field_1.fastq
Approx 75% complete for field_1.fastq
Approx 80% complete for field_1.fastq
Approx 85% complete for field_1.fastq
Approx 90% complete for field_1.fastq
Approx 95% complete for field_1.fastq
Analysis complete for field_1.fastq
Started analysis of field_2.fastq
Approx 5% complete for field_2.fastq
Approx 10% complete for field_2.fastq
Approx 15% complete for field_2.fastq
Approx 20% complete for field_2.fastq
Approx 25% complete for 

**Check results**
Write in the blan space the name of the file the .html extension. Disply in full screen. and then esc to come back to the notebook

In [4]:
import os
from IPython.core.display import display, HTML

# Ask the user for the file name they want to display
file_name = input("Enter the name of the HTML file you want to display (include .html extension): ")

# Check if the file exists
if os.path.exists(file_name):
    # Open and read the HTML file
    with open(file_name, 'r') as file:
        html_content = file.read()
        display(HTML(html_content))  # Display the HTML content
else:
    print(f"File '{file_name}' not found. Please ensure the file exists in the current directory.")


Enter the name of the HTML file you want to display (include .html extension): blan space
File 'blan space' not found. Please ensure the file exists in the current directory.


**Remove adapters and filter bad quality reads q > 20**

Load Trim galore
trim-galore will do the job

In [ ]:
%%bash
# 1️⃣  Install micromamba (lightweight conda)
wget -qO- https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba > /dev/null

# 2️⃣  Initialize the shell
eval "$(./bin/micromamba shell hook -s bash)"

# 3️⃣  Create an isolated environment (Python 3.12 avoids cutadapt conflict)
micromamba create -y -n tg -c conda-forge -c bioconda python=3.12 trim-galore cutadapt fastqc

# 4️⃣  Confirm installation
micromamba run -n tg trim_galore --version
micromamba run -n tg cutadapt --version


In [ ]:
%%bash
# Activate micromamba shell
eval "$(./bin/micromamba shell hook -s bash)"
micromamba run -n tg trim_galore --paired field_1.fastq field_2.fastq --clip_R1 15 --clip_R2 15 --length 80
# Run Trim Galore on a test file (change names as needed)
#micromamba run -n tg trim_galore --help
# Example real run:
# micromamba run -n tg trim_galore -q 20 --fastqc reads.fastq -o output_dir


**Run QC to the new validate reads. XXX_val_X.fq**

In [ ]:
!fastqc *.fq

**Check the quality control results**

In [ ]:
import os
from IPython.core.display import display, HTML

# Ask the user for the file name they want to display
file_name = input("Enter the name of the HTML file you want to display (include .html extension): ")

# Check if the file exists
if os.path.exists(file_name):
    # Open and read the HTML file
    with open(file_name, 'r') as file:
        html_content = file.read()
        display(HTML(html_content))  # Display the HTML content
else:
    print(f"File '{file_name}' not found. Please ensure the file exists in the current directory.")

**Align the Reads to a Reference Genome to Conduct a Guided Assembly by Mappi**ng

Check the available complete genomes in NCBI and download them to use as guide reference genomes.


In [ ]:
!datasets summary genome taxon "East African cassava mosaic virus" --assembly-level complete \
--as-json-lines | dataformat tsv genome --fields accession,organism-name,annotinfo-name
!datasets download genome accession GCF_000859785.1 --include gff3,rna,cds,protein,genome,seq-report
!unzip ncbi_dataset.zip

Copy the genome sequence in your current folder

In [ ]:
!cp ncbi_dataset/data/GCF_000859785.1/GCF_000859785.1_ViralMultiSegProj15177_genomic.fna ./

**Format the Reference Genome for Use as a Guide**

In [ ]:
!bwa index GCF_000859785.1_ViralMultiSegProj15177_genomic.fna

**Map the reads over the reference**

Mapping results are in field_isolated.sam

In [ ]:
!bwa mem GCF_000859785.1_ViralMultiSegProj15177_genomic.fna field_1_val_1.fq field_2_val_2.fq > field_isolated.sam

**Transform the results in sam format to bam format**

These comands will transform field_isolated.sam in field_isolated.bam, a compacted version

In [ ]:
!samtools view -bS field_isolated.sam > field_isolated.bam
!samtools sort field_isolated.bam -o field_isolated.bam
!samtools index field_isolated.bam

**Create a consensus and produce and assembled molecule**

The name of the assembled genome is field_isolated_consensus.fasta

In [ ]:
#!pilon --help
!pilon --genome GCF_000859785.1_ViralMultiSegProj15177_genomic.fna --frags field_isolated.bam --output field_isolated_consensus --vcf --changes

The generated results include an assembly of your reads based on the reference genome of the cassava virus obtained from NCBI. The number of corrections or polishing steps with Pilon could be important to investigate. Performing more than one iteration is recommended until no further changes are observed. For example, the code below assembles the reads based on the reference, creates the consensus, and runs Pilon two more times for additional polishing.

Input the files

In [ ]:
# Request file names from the user
REFERENCE = input("Enter the file name for the initial reference genome (e.g., initial_assembly.fasta): ")
READS_R1 = input("Enter the file name for the first pair of reads (e.g., illumina_reads_R1.fastq): ")
READS_R2 = input("Enter the file name for the second pair of reads (e.g., illumina_reads_R2.fastq): ")

# Output prefix and other settings
OUTPUT_PREFIX = "polished_genome"  # Prefix for output files
THREADS = 2  # Number of threads for bwa and Samtools

Enter the file name for the initial reference genome (e.g., initial_assembly.fasta): GCF_000859785.1_ViralMultiSegProj15177_genomic.fna
Enter the file name for the first pair of reads (e.g., illumina_reads_R1.fastq): field_1_val_1.fq
Enter the file name for the second pair of reads (e.g., illumina_reads_R2.fastq): field_2_val_2.fq


Run the pipeline

In [ ]:
import os

# Step 1: Initial Correction using the Original Reference
print("Starting initial Pilon correction with the original reference")

# Index the original reference for bwa
!bwa index "$REFERENCE"

# Align reads to the original reference
!bwa mem "$REFERENCE" "$READS_R1" "$READS_R2" > aligned_reads.sam

# Convert SAM to sorted BAM and index it
!samtools view -Sb aligned_reads.sam | samtools sort -@ {THREADS} -o sorted_reads.bam
!samtools index sorted_reads.bam

# Run Pilon with the sorted BAM file for the initial correction
initial_output = f"{OUTPUT_PREFIX}_initial"
!pilon --genome "$REFERENCE" --frags sorted_reads.bam --output "$initial_output" --changes

# Set the first corrected reference for the looped corrections
current_reference = f"{initial_output}.fasta"

# Step 2: Looped Corrections Starting with First Corrected Output
for i in range(1, 3):
    print(f"Starting Pilon iteration {i} with the corrected reference")

    # Index the current reference for bwa
    !bwa index "$current_reference"

    # Align reads to the current reference
    !bwa mem "$current_reference" "$READS_R1" "$READS_R2" > aligned_reads.sam

    # Convert SAM to sorted BAM and index it
    !samtools view -Sb aligned_reads.sam | samtools sort -@ {THREADS} -o sorted_reads.bam
    !samtools index sorted_reads.bam

    # Run Pilon with the sorted BAM file
    pilon_output = f"{OUTPUT_PREFIX}_iter_{i}"
    !pilon --genome "$current_reference" --frags sorted_reads.bam --output "$pilon_output" --changes

    # Update the reference for the next iteration
    current_reference = f"{pilon_output}.fasta"

print("Pilon polishing complete after initial correction and 2 additional iterations.")


#*De Novo* Assembly#
De novo assembly pieces together DNA fragments to form contigs that represent an organism's chromosomes. This approach assembles a genome from scratch, without relying on a reference genome or prior knowledge of the DNA sequence.

Let’s conduct a de novo assembly using SPAdes with the metagenome option.

Run the SPAdes assembler with the metagenome (--meta) option. We use the --meta option because the reads may still contain sequences from more than one isolate or multiple types of viruses. However, SPAdes offers other assembly options that you can explore depending on the characteristics of your data. For example:

Standard SPAdes: Use for single-organism assemblies when contamination is minimal.

Plasmid SPAdes (--plasmid): Optimized for plasmid assembly.
Hybrid SPAdes (--trusted-contigs): Combines long-read and short-read data for improved assembly.
RNA-Seq SPAdes (--isolate): This flag is highly recommended for high-coverage isolate and multi-cell Illumina data; improves the assembly quality and running time

Feel free to experiment with these options to find the best fit for your dataset

https://ablab.github.io/spades/

Get Spades assembler

In [ ]:
#2022-10-21 Steven Tang
#!wget http://cab.spbu.ru/files/release3.9.0/SPAdes-3.9.0-Linux.tar.gz
#!tar -xzf SPAdes-3.9.0-Linux.tar.gz

#2023-09-15 Renald Legaspi
#Updated: Spades3.9 to 3.15 since that version no longer runs on colab because a different version of python is being implemented.
#Fix: No longer installs the Linux tarfile due to segment fault issue. Spades is now being compiled from source.
# !wget http://cab.spbu.ru/files/release3.15.5/SPAdes-3.15.5.tar.gz
# !tar -xzf SPAdes-3.15.5.tar.gz
# !cd SPAdes-3.15.5
# !./SPAdes-3.15.5/spades_compile.sh

#2023-09-18 Steven Tang
#Fix: Use precompiled SPAdes that works with Colab
!wget https://github.com/steventango/colab-spades/releases/download/v3.15.5/SPAdes-3.15.5-Colab.tar.gz
!tar -xzf SPAdes-3.15.5-Colab.tar.gz


from datetime import datetime
from google.colab import files
from pathlib import Path
import subprocess

Now Lets assembly with spades

In [ ]:
# Tries to reduce the number of mismatches and short indels.
# Also runs MismatchCorrector: A post processing tool that uses BWA tool.
# Recommended mostly for small and/or low complexity genome.

#2022-10-21 Steven Tang
#careful_mode = True

#2023-09-15 Renald Legaspi
#Updated: Careful mode may cause the spades.py to crash due to insufficient RAM
careful_mode = False

#2023-09-15 Renald Legaspi
#Colab no longer implements python2; thus 'python /path/spades.py' is used instead of 'python2 /path/spades.py'

pe1_filename = "field_1_val_1.fq"
pe2_filename = "field_2_val_2.fq"
output_directory = f"{Path(pe1_filename).stem}_{Path(pe2_filename).stem}_{datetime.now().isoformat()}"

process = subprocess.run(
    f'python ./bin/spades.py -1 field_1_val_1.fq -2 field_2_val_2.fq -o field_meta_de_novo',
    capture_output=True,
    text=True,
    shell=True
)

#print(process.stdout)
#print(process.stderr)

The output of the assembly will be in the field_meta_de_novo_ folder. This folder contains several files, but you will need the contigs.fasta file. The assembly is located here.

Assess the metrics of the assembly using QUAST.

In [ ]:
!quast field_meta_de_novo/contigs.fasta

Check the report. You will find the number of contigs. Two contigs are larger than 1 Kb. These are the assemble viral genome.  

In [ ]:
!cat quast_results/latest/report.txt

#Comparision of two type of approaches
Comparing guided assembly with Pilon (using an existing reference to refine a new assembly) and de novo assembly (assembling a genome from scratch) brings both advantages and challenges. Let’s break down the pros and cons of each approach, as well as key post-analysis steps to consider for both.

Advantages and Disadvantages
1. Guided Assembly with Pilon (Reference-Guided)
Advantages:
Higher accuracy in correcting errors: Pilon leverages high-confidence reads (like Illumina) to polish assemblies, making it especially useful when using long-read assemblies (e.g., from Nanopore or PacBio) that may have higher base-calling errors.
Consistency with known reference: Using a reference genome ensures that conserved regions align well with established knowledge, making it easier to identify variant regions or perform comparative analyses.
Efficient for closely related species: When the target species is similar to the reference genome, reference-guided assembly can achieve high coverage with fewer ambiguities.
Disadvantages:
Bias towards the reference genome: Guided assembly may miss novel regions that are absent in the reference genome or over-represent reference biases, particularly in divergent regions.
Limited structural rearrangement detection: Large structural variants or genome rearrangements may be overlooked if the reference genome is heavily relied upon, as it constrains the assembly to follow reference structure.
2. De Novo Assembly
Advantages:

Discovery of novel sequences: De novo assembly can capture previously unknown or divergent regions that may not be present in any reference, which is critical for studying novel organisms or highly variable genomic regions.
More flexible structure: It provides a comprehensive picture of the genome without bias toward any reference, allowing for detection of structural variants, unique regions, and species-specific features.
Applicability to highly divergent species: When the target organism is distantly related to available references, de novo assembly avoids reference bias, allowing a more authentic genome reconstruction.
Disadvantages:

Higher computational and data demands: De novo assembly typically requires more computational resources and high-coverage sequencing data, especially for large genomes.
Increased assembly errors in repetitive regions: Without a reference to guide repetitive regions, it can be difficult to resolve these areas accurately, leading to fragmented or misassembled contigs.
Lower accuracy without polishing: De novo assemblies, especially those using long-read technologies, often require extensive polishing with high-confidence reads to achieve high accuracy.


You can run quast in the assembly results and compare